# SI Figure S8 (panel A): explicit-solvent correction vs solvent-shell size (vomicine)

The MagNET-x explicit-solvent correction (solvated minus isolated, averaged over a proton site's
atoms and MD frames) vs solvent-shell size (heavy-atom count of the solute-plus-solvent cluster),
for vomicine, one panel per solvent.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/applications", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import matplotlib.pyplot as plt

from applications_reader import Applications, SHELL_SIZES
import applications
import paths

In [ ]:
DATA = os.path.join(REPO, "data", "applications")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
loader = Applications(paths.dataset_file("applications", root=REPO),
                      os.path.join(DATA, "applications_experimental.xlsx"))

# compute the per-site, per-solvent shell-convergence corrections for vomicine (proton sites)
SOLUTE, NUCLEUS = "vomicine", "H"
corrections = applications.shell_convergence_corrections(loader, SOLUTE, NUCLEUS)
corrections.head()

In [ ]:
# panel layout and font styling for the 2x2 grid (one panel per solvent)
solvent_order = ["chloroform", "methanol", "TIP4P", "benzene"]
shell_cols = [f"shell_{s}" for s in SHELL_SIZES]
plt.rcParams.update({"font.family": "serif", "font.serif": ["Arial", "DejaVu Serif"],
                     "axes.labelsize": 12, "axes.titlesize": 14})

In [ ]:
# 2x2 panel, one solvent each; one line per proton site
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
for ax, solvent in zip(axes.ravel(), solvent_order):
    sub = corrections.xs(solvent, level="solvent")
    sub = sub.dropna(how="all")
    ax.set_title("TIP4P" if solvent == "TIP4P" else solvent.capitalize(), fontweight="bold")
    for site, row in sub.iterrows():
        ax.plot(SHELL_SIZES, row[shell_cols].values.astype(float),
                marker="o", linewidth=1.5, alpha=0.75, label=site)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Shell Size (Heavy Atom Count)", fontweight="bold")
    ax.set_ylabel("OpenMM correction (ppm)", fontweight="bold")
fig.suptitle("Shell-size Convergence for Vomicine", fontweight="bold", fontsize=18)
fig.tight_layout()
fig.savefig(figure_path("si_figure_s08_vomicine.png"), dpi=300, bbox_inches="tight")
plt.show()